In [2]:
# Directly index rows of W
# bigram neural net version but faster/optimized

import torch
import torch.nn.functional as F

In [3]:

words = open("names.txt", "r").read().splitlines()

chars = sorted(list(set("".join(words))))
stoi = {ch: i + 1 for i, ch in enumerate(chars)}
stoi["."] = 0
itos = {i: ch for ch, i in stoi.items()}

VOCAB_SIZE = len(stoi)

# input char -> next char


xs = []
ys = []

for word in words:
    
    chs = ["."] + list(word) + ["."]
    
    for ch1, ch2 in zip(chs, chs[1:]):
        xs.append(stoi[ch1])
        ys.append(stoi[ch2])

xs = torch.tensor(xs)
ys = torch.tensor(ys)

num_examples = xs.nelement()

print("training pairs:", num_examples)

training pairs: 228146


In [4]:
g = torch.Generator().manual_seed(2147483647)

W = torch.randn((VOCAB_SIZE, VOCAB_SIZE), generator=g, requires_grad=True)

In [5]:
for step in range(100):

    # logits = xenc @ W
    # directly select rows from W instead of xenc = F.one_hot(xs, num_classes=27).float()

    logits = W[xs]              

    counts = logits.exp()
    probs = counts / counts.sum(1, keepdim=True)

    loss = -probs[torch.arange(num_examples), ys].log().mean()


    loss = loss + 0.01 * (W ** 2).mean()


    W.grad = None
    loss.backward()

    W.data += -50 * W.grad

    if step % 10 == 0:
        print(f"step {step:3d} | loss {loss.item():.4f}")


step   0 | loss 3.7686
step  10 | loss 2.6965
step  20 | loss 2.5823
step  30 | loss 2.5414
step  40 | loss 2.5213
step  50 | loss 2.5099
step  60 | loss 2.5027
step  70 | loss 2.4979
step  80 | loss 2.4944
step  90 | loss 2.4919


In [6]:
print("\nGenerated names:\n")

for _ in range(10):

    out = []
    ix = 0

    while True:

        logits = W[ix]              # row lookup
        counts = logits.exp()
        p = counts / counts.sum()

        ix = torch.multinomial(
            p,
            num_samples=1,
            replacement=True,
            generator=g
        ).item()

        if ix == 0:
            break

        out.append(itos[ix])

    print("".join(out))


Generated names:

jigla
sadryropriniydavesole
rish
be
ka
nn
jo
s
t
a
